In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**CATBOOST**

In [1]:
!pip install catboost optuna -q

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier

import optuna

import warnings
warnings.filterwarnings('ignore')

In [11]:
import pandas as pd

file_path = '/kaggle/input/datasets/samrariaz123/dataset/eclipse/eclipse/bug-metrics.csv'

DF = pd.read_csv(file_path, sep=';')

print("Dataset Shape:", DF.shape)

DF.head()

Dataset Shape: (997, 12)


,classname,numberOfBugsFoundUntil:,numberOfNonTrivialBugsFoundUntil:,numberOfMajorBugsFoundUntil:,numberOfCriticalBugsFoundUntil:,numberOfHighPriorityBugsFoundUntil:,bugs,nonTrivialBugs,majorBugs,criticalBugs,highPriorityBugs,
0,org::eclipse::jdt::internal::core::search::ind...,3,2,0,0,0,0,0,0,0,0,
1,org::eclipse::jdt::internal::compiler::codegen...,0,0,0,0,0,0,0,0,0,0,
2,org::eclipse::jdt::internal::compiler::ast::AS...,55,48,6,4,2,1,0,0,0,0,
3,org::eclipse::jdt::internal::compiler::lookup:...,3,3,0,0,0,0,0,0,0,0,
4,org::eclipse::jdt::internal::eval::CodeSnippet...,15,13,1,1,0,0,0,0,0,0,


In [12]:
print(DF.columns)

Index(['classname ', ' numberOfBugsFoundUntil: ',
       ' numberOfNonTrivialBugsFoundUntil: ', ' numberOfMajorBugsFoundUntil: ',
       ' numberOfCriticalBugsFoundUntil: ',
       ' numberOfHighPriorityBugsFoundUntil: ', ' bugs ', ' nonTrivialBugs ',
       ' majorBugs ', ' criticalBugs ', ' highPriorityBugs ', ' '],
      dtype='object')


In [13]:
# Remove duplicate rows
DF = DF.drop_duplicates()

# Check missing values
print(DF.isnull().sum())

classname                                0
 numberOfBugsFoundUntil:                 0
 numberOfNonTrivialBugsFoundUntil:       0
 numberOfMajorBugsFoundUntil:            0
 numberOfCriticalBugsFoundUntil:         0
 numberOfHighPriorityBugsFoundUntil:     0
 bugs                                    0
 nonTrivialBugs                          0
 majorBugs                               0
 criticalBugs                            0
 highPriorityBugs                        0
                                         0
dtype: int64


In [15]:
print(DF.columns.tolist())

['classname ', ' numberOfBugsFoundUntil: ', ' numberOfNonTrivialBugsFoundUntil: ', ' numberOfMajorBugsFoundUntil: ', ' numberOfCriticalBugsFoundUntil: ', ' numberOfHighPriorityBugsFoundUntil: ', ' bugs ', ' nonTrivialBugs ', ' majorBugs ', ' criticalBugs ', ' highPriorityBugs ', ' ']


In [16]:
TARGET_COLUMN = 'bugs'

# Remove spaces from column names
DF.columns = DF.columns.str.strip()

X = DF.drop(TARGET_COLUMN, axis=1)

# Convert bug counts to binary classification
y = (DF[TARGET_COLUMN] > 0).astype(int)

print(X.shape)
print(y.value_counts())

(997, 11)
bugs
0    791
1    206
Name: count, dtype: int64


In [17]:
X = X.drop('classname', axis=1)

In [18]:
TARGET_COLUMN = 'bugs'

DF.columns = DF.columns.str.strip()

X = DF.drop(TARGET_COLUMN, axis=1)

# Remove text column
X = X.drop('classname', axis=1)

# Binary classification
y = (DF[TARGET_COLUMN] > 0).astype(int)

print(X.shape)
print(y.value_counts())

(997, 10)
bugs
0    791
1    206
Name: count, dtype: int64


In [19]:
# Convert categorical columns automatically
categorical_cols = X.select_dtypes(include=['object']).columns

for col in categorical_cols:
    X[col] = X[col].astype('category').cat.codes

print('Categorical columns encoded successfully')

Categorical columns encoded successfully


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(797, 10)
(200, 10)


In [21]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

X_train_scaled = numeric_transformer.fit_transform(X_train)
X_test_scaled = numeric_transformer.transform(X_test)

In [22]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

cat_model.fit(X_train_scaled, y_train)

cat_preds = cat_model.predict(X_test_scaled)
cat_probs = cat_model.predict_proba(X_test_scaled)[:,1]

cat_acc = accuracy_score(y_test, cat_preds)
cat_auc = roc_auc_score(y_test, cat_probs)

print('CatBoost Accuracy:', cat_acc)
print('CatBoost ROC AUC:', cat_auc)

CatBoost Accuracy: 0.86
CatBoost ROC AUC: 0.7767295597484276


In [23]:
extra_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

extra_model.fit(X_train_scaled, y_train)

extra_preds = extra_model.predict(X_test_scaled)
extra_probs = extra_model.predict_proba(X_test_scaled)[:,1]

extra_acc = accuracy_score(y_test, extra_preds)
extra_auc = roc_auc_score(y_test, extra_probs)

print('Extra Trees Accuracy:', extra_acc)
print('Extra Trees ROC AUC:', extra_auc)

Extra Trees Accuracy: 0.84
Extra Trees ROC AUC: 0.7950605921153552


In [24]:
base_models = [
    ('catboost', CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_seed=42
    )),

    ('extratrees', ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
]
meta_model = LogisticRegression()

stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train_scaled, y_train)

stack_preds = stack_model.predict(X_test_scaled)
stack_probs = stack_model.predict_proba(X_test_scaled)[:,1]

stack_acc = accuracy_score(y_test, stack_preds)
stack_auc = roc_auc_score(y_test, stack_probs)

print('Stacking Accuracy:', stack_acc)
print('Stacking ROC AUC:', stack_auc)

Stacking Accuracy: 0.85
Stacking ROC AUC: 0.8064887252646111


In [25]:
print(classification_report(y_test, stack_preds))

              precision    recall  f1-score   support

           0       0.86      0.97      0.91       159
           1       0.76      0.39      0.52        41

    accuracy                           0.85       200
   macro avg       0.81      0.68      0.71       200
weighted avg       0.84      0.85      0.83       200



In [26]:
cm = confusion_matrix(y_test, stack_preds)
print(cm)

[[154   5]
 [ 25  16]]


In [27]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': cat_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
0,numberOfBugsFoundUntil:,29.244299
1,numberOfNonTrivialBugsFoundUntil:,27.999857
2,numberOfMajorBugsFoundUntil:,14.887408
4,numberOfHighPriorityBugsFoundUntil:,10.366677
3,numberOfCriticalBugsFoundUntil:,9.162225
6,majorBugs,6.368061
5,nonTrivialBugs,1.490478
7,criticalBugs,0.449261
8,highPriorityBugs,0.031734
9,,0.000000


In [29]:
def objective(trial):

    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'verbose': 0,
        'random_seed': 42
    }

    model = CatBoostClassifier(**params)

    model.fit(X_train_scaled, y_train)

    preds = model.predict_proba(X_test_scaled)[:, 1]

    auc = roc_auc_score(y_test, preds)

    return auc


study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=20)

print('Best Parameters:', study.best_params)

print('Best ROC AUC:', study.best_value)

[I 2026-05-10 09:57:49,662] A new study created in memory with name: no-name-dba1ca5c-e067-4073-a4ae-26d3e9639177
[I 2026-05-10 09:57:50,117] Trial 0 finished with value: 0.7718208314158614 and parameters: {'iterations': 214, 'depth': 9, 'learning_rate': 0.2587997104265984, 'l2_leaf_reg': 7.546521241199972}. Best is trial 0 with value: 0.7718208314158614.
[I 2026-05-10 09:57:50,837] Trial 1 finished with value: 0.7442092345451757 and parameters: {'iterations': 884, 'depth': 4, 'learning_rate': 0.12281778415069841, 'l2_leaf_reg': 2.009109184738402}. Best is trial 0 with value: 0.7718208314158614.
[I 2026-05-10 09:57:52,674] Trial 2 finished with value: 0.7782635373523547 and parameters: {'iterations': 928, 'depth': 9, 'learning_rate': 0.028148394507972796, 'l2_leaf_reg': 4.846712046086395}. Best is trial 2 with value: 0.7782635373523547.
[I 2026-05-10 09:57:53,371] Trial 3 finished with value: 0.762310170271514 and parameters: {'iterations': 575, 'depth': 7, 'learning_rate': 0.188345951

Best Parameters: {'iterations': 438, 'depth': 7, 'learning_rate': 0.010168190128480081, 'l2_leaf_reg': 9.996345019097364}
Best ROC AUC: 0.8110906580763921


In [30]:
best_params = study.best_params

final_model = CatBoostClassifier(
    **best_params,
    verbose=0,
    random_seed=42
)

final_model.fit(X_train_scaled, y_train)

final_preds = final_model.predict(X_test_scaled)
final_probs = final_model.predict_proba(X_test_scaled)[:,1]

final_acc = accuracy_score(y_test, final_preds)
final_auc = roc_auc_score(y_test, final_probs)

print('FINAL ACCURACY:', final_acc)
print('FINAL ROC AUC:', final_auc)

FINAL ACCURACY: 0.865
FINAL ROC AUC: 0.8110906580763921
